# Chunking Strategies, Side by Side

Before text can be embedded and retrieved, it has to be cut into **chunks**.
The way you cut matters a lot:

- Chunks that are **too big** dilute the meaning and blow past context limits.
- Chunks that are **too small** lose the surrounding context that makes them answerable.
- Chunks cut at the **wrong place** (mid-word, mid-sentence) hand the model garble.

A bad chunk is the *first* domino: bad chunk -> bad embedding -> wrong neighbour
retrieved -> wrong answer. So it is worth knowing more than one way to do it.

This notebook builds up six strategies on the same passage of *The Adventures of
Sherlock Holmes* so they can be compared directly:

1. **Fixed-size** characters (the naive baseline)
2. **Sliding window** (fixed-size **with overlap**)
3. **Sentence-aware** (never cut mid-sentence)
4. **Recursive** (try big separators first, fall back to smaller ones)
5. **Hierarchical** (follow the document's own structure: chapters -> paragraphs)
6. **Semantic** (cut where the topic shifts)

Everything here is plain Python with no external models, so it runs anywhere.

## Setup — load the text

We read the book, strip the Project Gutenberg license header/footer, and keep one
clean passage as our running example. The same `sample` is fed to every method so
the comparisons are fair.

In [1]:
import re            # regular expressions — used for word counting and (later) heading detection
import statistics    # gives us statistics.mean() for the average chunk length

# The book lives next to this notebook in the corpus_jupyter/ folder.
PATH = "corpus_jupyter/Adventures_of_Sherlock_Holmes.txt"

# Read the entire file into one big string.
raw = open(PATH, encoding="utf-8").read()

# --- Strip the Project Gutenberg license boilerplate -------------------------
# Every Gutenberg book is wrapped in a legal header and footer. We only want the
# story itself, so we slice the string between the two "*** ... ***" markers.
#   split(marker, 1)[-1] -> everything AFTER the start marker
#   split(marker, 1)[0]  -> everything BEFORE the end marker
body = raw.split("*** START OF THE PROJECT GUTENBERG EBOOK", 1)[-1]
body = body.split("*** END OF THE PROJECT GUTENBERG EBOOK", 1)[0]

# --- Take ONE clean passage and UNWRAP it ------------------------------------
# Gutenberg hard-wraps lines at ~70 characters (a newline lands every ~70 chars,
# even in the middle of a sentence). A chunker should reason about logical
# paragraphs, not those arbitrary line breaks, so we rebuild clean paragraphs:
#   - find where the first story actually begins ("To Sherlock Holmes ...")
#   - split into paragraphs on blank lines ("\n\n")
#   - inside each paragraph, " ".join(p.split()) collapses every run of
#     whitespace (including the mid-sentence newlines) down to single spaces
start = body.find("To Sherlock Holmes")
paragraphs = [" ".join(p.split()) for p in body[start:].split("\n\n") if p.strip()]

# Keep just the first 7 paragraphs as our running example. A small, varied
# passage (a few long narrative paragraphs + some short ones) is enough to make
# every method behave visibly differently.
sample = "\n\n".join(paragraphs[:7])

print("whole book :", len(body), "characters")
print("Words in the book:", len(re.findall(r"\w+", body)))   # \w+ = runs of word characters
print("our sample :", len(sample), "characters,", sample.count("\n\n") + 1, "paragraphs\n")
print(sample[:300], "...")                                   # peek at the start of the sample

whole book : 562250 characters
Words in the book: 105960
our sample : 4081 characters, 7 paragraphs

To Sherlock Holmes she is always _the_ woman. I have seldom heard him mention her under any other name. In his eyes she eclipses and predominates the whole of her sex. It was not that he felt any emotion akin to love for Irene Adler. All emotions, and that one particularly, were abhorrent to his col ...


## A tiny toolkit we reuse everywhere

Two helpers, defined once, so every method below is just its own chunking logic:

- `show_chunks(chunks)` — print the first few chunks with their lengths.
- `chunk_stats(chunks)` — count, size spread, and a **boundary-quality** score:
  the share of chunks that end on real sentence punctuation (`.`, `!`, `?`).
  A high score means we rarely cut mid-thought — that single number tells most of
  the story when comparing methods.

In [2]:
# show_chunks: pretty-print the first few chunks so we can eyeball WHERE the cuts
# landed. n = how many chunks to show, width = how many characters of each to print.
def show_chunks(chunks, n=3, width=220):
    for i, c in enumerate(chunks[:n]):                 # only the first n chunks
        # Truncate very long chunks so the printout stays readable.
        shown = c if len(c) <= width else c[:width] + " ..."
        print(f"--- chunk {i}  (len {len(c)}) " + "-" * 28)
        print(shown.strip())
    if len(chunks) > n:                                # note how many we hid
        print(f"... (+{len(chunks) - n} more chunks)")

# chunk_stats: the numbers we use to compare one method against another.
def chunk_stats(chunks):
    lens = [len(c) for c in chunks]                    # the length of every chunk

    # BOUNDARY QUALITY: how many chunks end on real sentence punctuation?
    # rstrip() drops trailing spaces, then endswith((...)) checks the last char.
    # A chunk ending in "." or "?" almost certainly ended at a sentence; one
    # ending mid-word was cut badly. This single % tells most of the story.
    clean = sum(1 for c in chunks if c.rstrip().endswith((".", "!", "?", '"', "”")))

    print(f"chunks         : {len(chunks)}")
    print(f"size  min/avg/max: {min(lens)} / {int(statistics.mean(lens))} / {max(lens)}")
    print(f"clean endings  : {clean}/{len(chunks)} = {100 * clean // len(chunks)}%")

# One shared target size (in characters) so every method is comparable.
CHUNK_SIZE = 300

## 1. Fixed-size characters — the naive baseline

The simplest possible rule: every `CHUNK_SIZE` characters, cut. No awareness of
words or sentences at all.

It is fast and dead simple, but watch the chunk boundaries — they land in the
middle of words and sentences, and the boundary-quality score is poor.

In [3]:
# Walk the text in fixed jumps of CHUNK_SIZE and slice out each block.
#   range(0, len(sample), CHUNK_SIZE) -> 0, 300, 600, 900, ...
#   sample[i:i + CHUNK_SIZE]          -> the 300 characters starting at i
# No awareness of words or sentences at all — the cut lands wherever it lands.
chunks_fixed = [sample[i:i + CHUNK_SIZE] for i in range(0, len(sample), CHUNK_SIZE)]

show_chunks(chunks_fixed)
print()
chunk_stats(chunks_fixed)          # watch the clean-endings score: it will be tiny

--- chunk 0  (len 300) ----------------------------
To Sherlock Holmes she is always _the_ woman. I have seldom heard him mention her under any other name. In his eyes she eclipses and predominates the whole of her sex. It was not that he felt any emotion akin to love for ...
--- chunk 1  (len 300) ----------------------------
d, precise but admirably balanced mind. He was, I take it, the most perfect reasoning and observing machine that the world has seen, but as a lover he would have placed himself in a false position. He never spoke of the  ...
--- chunk 2  (len 300) ----------------------------
e observer—excellent for drawing the veil from men’s motives and actions. But for the trained reasoner to admit such intrusions into his own delicate and finely adjusted temperament was to introduce a distracting factor  ...
... (+11 more chunks)

chunks         : 14
size  min/avg/max: 181 / 291 / 300
clean endings  : 1/14 = 7%


## 2. Sliding window — fixed size **with overlap**

Same fixed cutting, but each chunk starts a little **before** the previous one
ended. That repeated `overlap` of text means a sentence split across a boundary
still appears whole inside at least one chunk — so a retriever can find it.

The cost: overlap duplicates text, so you store more chunks for the same book.

In [4]:
overlap = 60                       # how many characters each chunk shares with the previous one
step = CHUNK_SIZE - overlap        # so we ADVANCE only 240 chars but still GRAB 300

# Same slicing as fixed-size, but because step < CHUNK_SIZE each new chunk starts
# 60 chars before the previous one ended -> neighbouring chunks overlap.
chunks_window = [sample[i:i + CHUNK_SIZE] for i in range(0, len(sample), step)]

# Prove the overlap: the LAST 60 chars of chunk 0 equal the FIRST 60 of chunk 1.
# repr(...) shows the quotes/whitespace so the match is unambiguous.
print("tail of chunk 0:", repr(chunks_window[0][-overlap:]))
print("head of chunk 1:", repr(chunks_window[1][:overlap]))
print("--> the overlap is identical, so context survives the cut\n")

show_chunks(chunks_window)
print()
chunk_stats(chunks_window)         # more chunks than fixed-size: overlap = duplicated text

tail of chunk 0: 'otions, and that one particularly, were abhorrent to his col'
head of chunk 1: 'otions, and that one particularly, were abhorrent to his col'
--> the overlap is identical, so context survives the cut

--- chunk 0  (len 300) ----------------------------
To Sherlock Holmes she is always _the_ woman. I have seldom heard him mention her under any other name. In his eyes she eclipses and predominates the whole of her sex. It was not that he felt any emotion akin to love for ...
--- chunk 1  (len 300) ----------------------------
otions, and that one particularly, were abhorrent to his cold, precise but admirably balanced mind. He was, I take it, the most perfect reasoning and observing machine that the world has seen, but as a lover he would hav ...
--- chunk 2  (len 300) ----------------------------
a false position. He never spoke of the softer passions, save with a gibe and a sneer. They were admirable things for the observer—excellent for drawing the veil from men’s mo

## 3. Sentence-aware — never cut mid-sentence

Instead of counting characters, we first split the text into **sentences**, then
greedily pack whole sentences into a chunk until adding the next one would blow
the budget. Boundaries now always land between sentences.

To find the sentences we use a plain, readable trick instead of a regex: wherever
a `.`, `!`, or `?` is followed by a space, drop in a line break and split on those
line breaks. The punctuation stays attached, so every sentence keeps its full stop.
(The compact one-line regex that does the same thing is left as a comment in the
code, for anyone who wants it.)

Notice the boundary-quality score jump compared to the naive methods.

In [5]:
# Split the passage into sentences. We keep this deliberately simple and
# regex-free: put a line break after every sentence-ending mark that is followed
# by a space, then split on those line breaks. The punctuation stays attached,
# so each sentence keeps its full stop.
flat = sample.replace("\n", " ")                   # flatten paragraphs into one long line first
for end in [". ", "! ", "? "]:
    flat = flat.replace(end, end[0] + "\n")        # ". " -> ".\n", "! " -> "!\n", "? " -> "?\n"
sentences = [s.strip() for s in flat.split("\n") if s.strip()]   # split on the breaks, drop blanks

# The exact same result as this one-line regex, kept here for reference:
# sentences = re.split(r"(?<=[.!?])\s+", sample.replace("\n", " "))

# GREEDY PACKING: keep adding whole sentences to `current` until the next one
# would exceed CHUNK_SIZE; then flush `current` as a finished chunk and start over.
chunks_sentence = []
current = ""
for s in sentences:
    if len(current) + len(s) + 1 <= CHUNK_SIZE:    # +1 accounts for the space we join with
        current = (current + " " + s).strip()      # still room -> append this sentence
    else:
        if current:
            chunks_sentence.append(current)        # full -> store what we had so far
        current = s                                # start the next chunk with this sentence
if current:
    chunks_sentence.append(current)                # don't forget the last partial chunk

show_chunks(chunks_sentence)
print()
chunk_stats(chunks_sentence)                       # boundaries land between sentences -> 100%

--- chunk 0  (len 233) ----------------------------
To Sherlock Holmes she is always _the_ woman. I have seldom heard him mention her under any other name. In his eyes she eclipses and predominates the whole of her sex. It was not that he felt any emotion akin to love for ...
--- chunk 1  (len 263) ----------------------------
All emotions, and that one particularly, were abhorrent to his cold, precise but admirably balanced mind. He was, I take it, the most perfect reasoning and observing machine that the world has seen, but as a lover he wou ...
--- chunk 2  (len 175) ----------------------------
He never spoke of the softer passions, save with a gibe and a sneer. They were admirable things for the observer—excellent for drawing the veil from men’s motives and actions.
... (+14 more chunks)

chunks         : 17
size  min/avg/max: 84 / 238 / 473
clean endings  : 17/17 = 100%


## 4. Recursive — try big separators first, fall back to smaller ones

This is the idea behind LangChain's `RecursiveCharacterTextSplitter`. We keep an
ordered list of separators from coarse to fine:

```
paragraph break  ->  line break  ->  ". "  ->  space  ->  raw characters
```

Split on the **coarsest** separator first. Any piece still larger than the budget
gets re-split using the **next** separator down the list, and so on. The result
respects the largest natural boundary that fits — paragraphs stay whole when they
can, and only genuinely huge blobs get hard-cut.

Try to split on the biggest natural boundary first (paragraphs). If a piece is still too big, re-split that piece on the next-smaller boundary (lines, then sentences, then words, then raw characters). Recurse downward only as far as you must.

In [6]:
# A quick look at the "unpacking" trick the recursive splitter uses below.
# Separators run from COARSEST (paragraph break) to FINEST ("" = raw characters).
separators = ("\n\n", "\n", ". ", " ", "")

# sep, *rest  ->  sep is the FIRST separator, rest is the LIST of the remaining ones.
# Each recursion peels off the current `sep` and passes `rest` down a level.
sep, *rest = separators
print(repr(sep), repr(rest))

'\n\n' ['\n', '. ', ' ', '']


In [7]:
def recursive_split(text, size, separators=("\n\n", "\n", ". ", " ", "")):
    # BASE CASE: the piece already fits -> return it (ignore empty/whitespace).
    if len(text) <= size:
        return [text] if text.strip() else []

    sep, *rest = separators                        # current separator + the finer ones still to try
    if sep == "":                                  # last resort: no separators left -> hard char cut
        return [text[i:i + size] for i in range(0, len(text), size)]

    # Split on this separator but KEEP it attached to each piece, so a sentence
    # never loses its full stop (this is LangChain's keep_separator behaviour).
    #   "a. b. c".split(". ") -> ["a", "b", "c"]; we re-glue ". " onto all but the last.
    parts = text.split(sep)
    parts = [p + sep for p in parts[:-1]] + [parts[-1]]

    # Greedily pack the pieces back together up to `size`.
    chunks, buf = [], ""
    for part in parts:
        if len(buf) + len(part) <= size:
            buf += part                            # fits -> keep packing at this level
        elif len(part) > size:                     # one part ALONE is bigger than the budget...
            if buf:
                chunks.append(buf)                 # flush what we have, then
            chunks.extend(recursive_split(part, size, tuple(rest)))   # ...re-split it FINER
            buf = ""
        else:                                      # part fits, but not on top of the current buf
            if buf:
                chunks.append(buf)                 # flush, then start a fresh buffer with it
            buf = part
    if buf:
        chunks.append(buf)                         # flush the final buffer
    return [c.strip() for c in chunks if c.strip()]

chunks_recursive = recursive_split(sample, CHUNK_SIZE)

show_chunks(chunks_recursive)
print()
chunk_stats(chunks_recursive)

--- chunk 0  (len 233) ----------------------------
To Sherlock Holmes she is always _the_ woman. I have seldom heard him mention her under any other name. In his eyes she eclipses and predominates the whole of her sex. It was not that he felt any emotion akin to love for ...
--- chunk 1  (len 263) ----------------------------
All emotions, and that one particularly, were abhorrent to his cold, precise but admirably balanced mind. He was, I take it, the most perfect reasoning and observing machine that the world has seen, but as a lover he wou ...
--- chunk 2  (len 175) ----------------------------
He never spoke of the softer passions, save with a gibe and a sneer. They were admirable things for the observer—excellent for drawing the veil from men’s motives and actions.
... (+17 more chunks)

chunks         : 20
size  min/avg/max: 27 / 202 / 299
clean endings  : 18/20 = 90%


## 5. Hierarchical — follow the document's own structure

The methods above ignore that a book *already has* structure: chapters, then
paragraphs, then sentences. Hierarchical chunking uses that structure directly.

First we detect the twelve chapter headings (Roman numeral + ALL-CAPS title).
Each chapter becomes a **parent**; its paragraphs become **child** chunks.

In [8]:
# Match chapter headings like "I. A SCANDAL IN BOHEMIA" at the start of a line.
#   ^              -> start of a line (re.MULTILINE makes ^ match every line, not just the first)
#   ([IVXLCDM]+)   -> group 1: one or more Roman-numeral letters
#   \.             -> a literal dot right after the numeral
#   [ \t]+         -> spaces/tabs only (NOT \s, which would also swallow newlines)
#   (.+)$          -> group 2: the rest of the line = the title text
chapter_re = re.compile(r"^([IVXLCDM]+)\.[ \t]+(.+)$", re.MULTILINE)

# finditer finds every match; we keep only those whose title is ALL-CAPS, which
# filters out false positives like a stray "I. " appearing mid-paragraph.
headings = [m for m in chapter_re.finditer(body) if m.group(2).strip().isupper()]

# Chapter boundaries = where each heading starts, plus len(body) as a final sentinel
# so the last chapter has somewhere to stop.
bounds = [m.start() for m in headings] + [len(body)]
chapters = []
for i, m in enumerate(headings):
    title = f"{m.group(1)}. {m.group(2).strip()}"   # "I" + ". " + "A SCANDAL IN BOHEMIA"
    # A chapter's text runs from its own heading to the START of the next heading.
    content = body[m.start():bounds[i + 1]].strip()
    chapters.append((title, content))

print(f"detected {len(chapters)} chapters:\n")
for title, content in chapters:
    print(f"  {title:46}  {len(content):>6} chars")   # :46 left-pads, :>6 right-aligns the count

detected 12 chapters:

  I. A SCANDAL IN BOHEMIA                          46518 chars
  II. THE RED-HEADED LEAGUE                        49256 chars
  III. A CASE OF IDENTITY                          37916 chars
  IV. THE BOSCOMBE VALLEY MYSTERY                  51353 chars
  V. THE FIVE ORANGE PIPS                          39444 chars
  VI. THE MAN WITH THE TWISTED LIP                 49160 chars
  VII. THE ADVENTURE OF THE BLUE CARBUNCLE         42114 chars
  VIII. THE ADVENTURE OF THE SPECKLED BAND         52949 chars
  IX. THE ADVENTURE OF THE ENGINEER’S THUMB        44602 chars
  X. THE ADVENTURE OF THE NOBLE BACHELOR           44149 chars
  XI. THE ADVENTURE OF THE BERYL CORONET           51001 chars
  XII. THE ADVENTURE OF THE COPPER BEECHES         53136 chars


Now build the two-level tree for one chapter: the chapter is the parent, and its
paragraphs (split on blank lines) are the children.

In a real RAG system you **retrieve on the small child** (precise match) but
**feed the model the parent or neighbours** (full context) — best of both: a tight
search target without losing the surrounding story.

In [9]:
# Take the first chapter as the PARENT (the big, context-rich unit).
parent_title, parent_text = chapters[0]

# Its CHILDREN are the paragraphs inside it (split on blank lines). We skip tiny
# fragments (< 40 chars) like stray headings or page numbers.
children = [p.strip() for p in parent_text.split("\n\n") if len(p.strip()) > 40]

print(f"PARENT: {parent_title}  ({len(parent_text)} chars, {len(children)} paragraph children)\n")
for i, child in enumerate(children[:4]):            # preview just the first 4 children
    preview = child.replace("\n", " ")[:90]         # first 90 chars, on one line
    print(f"  child {i:2}  (len {len(child):>4}): {preview} ...")
print(f"  ... (+{len(children) - 4} more paragraphs)")

PARENT: I. A SCANDAL IN BOHEMIA  (46518 chars, 151 paragraph children)

  child  0  (len 1147): To Sherlock Holmes she is always _the_ woman. I have seldom heard him mention her under an ...
  child  1  (len 1300): I had seen little of Holmes lately. My marriage had drifted us away from each other. My ow ...
  child  2  (len  976): One night—it was on the twentieth of March, 1888—I was returning from a journey to a patie ...
  child  3  (len  337): His manner was not effusive. It seldom was; but he was glad, I think, to see me. With hard ...
  ... (+147 more paragraphs)


## 6. Semantic — cut where the topic shifts

The most adaptive idea: place boundaries where **meaning** changes, not at a fixed
length. Production systems embed each sentence and start a new chunk where the
similarity between neighbours drops.

To keep this runnable with no model, we use a lightweight stand-in for "are these
two sentences about the same thing": **word overlap** (Jaccard similarity). The
mechanism is identical — only the similarity measure is simpler. A real system
swaps the one line that computes `sim` for an embedding cosine similarity.

In [10]:
# A bag-of-words for a sentence: the set of lowercase word tokens it contains.
# [a-z']+ keeps letters and apostrophes (so "don't" stays a single word).
def word_set(s):
    return set(re.findall(r"[a-z']+", s.lower()))

sents = sentences                                  # reuse the sentence split from step 3

# Similarity between each adjacent pair of sentences (Jaccard = overlap / union).
#   wa & wb = words in BOTH sentences,  wa | wb = words in EITHER sentence.
#   ratio near 1 -> very similar; near 0 -> probably a topic change.
sims = []
for a, b in zip(sents, sents[1:]):
    wa, wb = word_set(a), word_set(b)
    sims.append(len(wa & wb) / len(wa | wb) if (wa | wb) else 0.0)

THRESHOLD = 0.06                                   # below this similarity = "topic shifted"

# Walk the sentences: keep gluing them together while neighbours are similar, and
# cut a new chunk whenever the link drops below THRESHOLD.
chunks_semantic = []
current = sents[0]                                 # the first chunk starts with sentence 0
for sent, sim in zip(sents[1:], sims):
    if sim < THRESHOLD:                            # weak link -> start a new chunk
        chunks_semantic.append(current)
        current = sent
    else:
        current = current + " " + sent             # strong link -> keep them in one chunk
chunks_semantic.append(current)                    # flush the final chunk

print("adjacent-sentence similarity (lower = bigger topic shift):")
print("  " + "  ".join(f"{s:.2f}" for s in sims), "\n")
show_chunks(chunks_semantic)
print()
chunk_stats(chunks_semantic)

adjacent-sentence similarity (lower = bigger topic shift):
  0.00  0.05  0.00  0.07  0.07  0.11  0.07  0.07  0.08  0.05  0.04  0.07  0.03  0.08  0.15  0.10  0.08  0.11  0.15  0.08  0.06  0.00  0.10  0.12  0.05  0.07  0.12  0.17  0.05  0.05  0.14  0.25  0.08  0.00 

--- chunk 0  (len 45) ----------------------------
To Sherlock Holmes she is always _the_ woman.
--- chunk 1  (len 57) ----------------------------
I have seldom heard him mention her under any other name.
--- chunk 2  (len 63) ----------------------------
In his eyes she eclipses and predominates the whole of her sex.
... (+9 more chunks)

chunks         : 12
size  min/avg/max: 33 / 338 / 1990
clean endings  : 12/12 = 100%


## All six, side by side

Every method ran on the **same** passage. Comparing `#chunks`, average length,
and the clean-ending score makes the trade-offs concrete.

In [11]:
# For the table, represent "hierarchical" with the sample's own paragraphs (the
# child level), since that's the unit a hierarchical splitter would emit here.
chunks_hier = [p.strip() for p in sample.split("\n\n") if p.strip()]

# Pair each method's name with the chunk list we built earlier in the notebook.
methods = [
    ("1. fixed (no overlap)", chunks_fixed),
    ("2. sliding window",     chunks_window),
    ("3. sentence-aware",     chunks_sentence),
    ("4. recursive",          chunks_recursive),
    ("5. hierarchical (para)",chunks_hier),
    ("6. semantic",           chunks_semantic),
]

# Print one aligned row per method. The format specs (:24, :>9, :>11) pad each
# column to a fixed width so the table lines up neatly.
print(f"{'method':24}{'#chunks':>9}{'avg len':>9}{'clean end':>11}")
print("-" * 53)
for name, ch in methods:
    lens = [len(c) for c in ch]
    clean = sum(1 for c in ch if c.rstrip().endswith((".", "!", "?", '"', "”")))
    print(f"{name:24}{len(ch):>9}{int(statistics.mean(lens)):>9}{str(100 * clean // len(ch)) + '%':>11}")

method                    #chunks  avg len  clean end
-----------------------------------------------------
1. fixed (no overlap)          14      291         7%
2. sliding window              18      280        11%
3. sentence-aware              17      238       100%
4. recursive                   20      202        90%
5. hierarchical (para)          7      581       100%
6. semantic                    12      338       100%


## Which one should you reach for?

| Strategy | Respects meaning? | Overlap? | Needs structure? | Reach for it when... |
|---|---|---|---|---|
| Fixed-size | No | No | No | you need something in five minutes; data is uniform |
| Sliding window | No | **Yes** | No | answers straddle boundaries and recall matters |
| Sentence-aware | Sentences | Optional | No | clean prose; you want readable, self-contained chunks |
| Recursive | Best-effort | Optional | No | **mixed content** (code, lists, prose) — the safe default |
| Hierarchical | Yes | No | **Yes** | the document has real structure (chapters, headings, sections) |
| Semantic | **Yes** | No | No | topic drifts a lot; you can afford to embed while chunking |

**Takeaways**

- The naive methods are cheap but score worst on clean boundaries — and boundaries
  are exactly what retrieval quality hinges on.
- **Recursive** is the pragmatic default: structure-aware enough for most text,
  with no model and no schema required.
- **Hierarchical** wins when the document hands you structure for free — and
  parent/child chunks let you search small but answer with full context.
- **Semantic** is the most adaptive but the most expensive; reach for it when topic
  drift, not length, is your real problem.